<a href="https://colab.research.google.com/github/cybercolombia/suelosabio/blob/dev/notebooks/MeteoData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import geopandas as gpd
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point
from shapely.geometry.polygon import Polygon
from matplotlib.colors import ListedColormap

In [ ]:
estaciones_df= pd.read_csv('/content/drive/MyDrive/eco2026/Estaciones_IDEAM_20260527.csv')

In [ ]:
estaciones_df.info()

In [ ]:
estaciones_df.head()

In [ ]:
print(estaciones_df['Departamento'].unique())

In [ ]:
dptos={'Cundinamarca',''}

In [ ]:
estaciones_df['Estado'].value_counts()

In [ ]:
departamentos_interes = ['Antioquia', 'Boyacá', 'Cundinamarca']
estaciones_activas_filtradas_df = estaciones_df[
    (estaciones_df['Estado'] == 'Activa') &
    (estaciones_df['Departamento'].isin(departamentos_interes))
]

In [ ]:
print(f"Número de estaciones filtradas: {len(estaciones_activas_filtradas_df)}")

In [ ]:
estaciones_activas_filtradas_df['Departamento'].value_counts()

In [ ]:
estaciones_activas_filtradas_df.to_csv('/content/drive/MyDrive/eco2026/EstacionesActivasIDEAM.csv')

In [ ]:
estaciones_activas=estaciones_activas_filtradas_df['Codigo'].unique()

In [ ]:
datos_estaciones= pd.read_csv('/content/drive/MyDrive/eco2026/Datos_Estaciones_20260527.csv')

In [ ]:
datos_estaciones.info()

In [ ]:
datos_estaciones.head()

In [ ]:
datos_estaciones_filtradas_df = datos_estaciones[
    (datos_estaciones['Departamento'].isin(departamentos_interes))
]

In [ ]:
unique_codigo_estacion = datos_estaciones_filtradas_df['CodigoEstacion'].unique()

In [ ]:
print(len(estaciones_activas))
print(len(unique_codigo_estacion))

In [ ]:
original_count = len(datos_estaciones_filtradas_df)
datos_estaciones_filtradas_df['FechaObservacion'] = pd.to_datetime(datos_estaciones_filtradas_df['FechaObservacion'], errors='coerce')

converted_count = datos_estaciones_filtradas_df['FechaObservacion'].count()
error_count = original_count - converted_count

print(f"Total de registros: {original_count}")
print(f"Registros transformados exitosamente: {converted_count}")
print(f"Registros con error de transformación: {error_count}")


In [ ]:
datos_estaciones_filtradas_df.info()

In [ ]:
primera_fecha_observacion = datos_estaciones_filtradas_df['FechaObservacion'].min()
ultima_fecha_observacion = datos_estaciones_filtradas_df['FechaObservacion'].max()

print(f"Primera fecha de observación: {primera_fecha_observacion}")
print(f"Última fecha de observación: {ultima_fecha_observacion}")

In [ ]:
import requests
import pandas as pd
from datetime import datetime

def consultar_temperatura_aire(municipio=None, fecha_inicio=None, fecha_fin=None, limit=50000):
    """
    Consulta datos de Temperatura Ambiente del Aire

    Args:
        municipio: str (ej: 'Bogotá')
        fecha_inicio: str 'YYYY-MM-DD' (ej: '2026-01-01')
        fecha_fin: str 'YYYY-MM-DD' (ej: '2026-06-04')
    """
    dataset_id = "sbwg-7ju4"
    url = f"https://www.datos.gov.co/resource/{dataset_id}.json"

    conditions = []

    if municipio is not None:
        conditions.append(f"`Municipio` = '{municipio}'")

    # Usar >= y <= en lugar de BETWEEN
    if fecha_inicio is not None and fecha_fin is not None:
        try:
            inicio = datetime.strptime(fecha_inicio, '%Y-%m-%d')
            fin = datetime.strptime(fecha_fin, '%Y-%m-%d')

            inicio_str = inicio.strftime('%Y %b %d 12:00:00 AM')
            fin_str = fin.strftime('%Y %b %d 11:59:59 PM')

            # Cambio: usar >= y <= en lugar de BETWEEN
            conditions.append(f"`FechaObservacion` >= '{inicio_str}'")
            conditions.append(f"`FechaObservacion` <= '{fin_str}'")

            print(f"Rango: {inicio_str} a {fin_str}")

        except ValueError as e:
            print(f"Error en formato fecha: {e}")
            return pd.DataFrame()

    where_clause = " AND ".join(conditions) if conditions else ""
    where_str = f"WHERE {where_clause}" if where_clause else ""

    query = f"""
    SELECT *
    {where_str}
    LIMIT {limit}
    """

    print(f"Consultando: municipio={municipio}")

    params = {'$query': query}

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()

        data = response.json()

        if len(data) == 0:
            print(f"0 registros encontrados")
            return pd.DataFrame()

        df = pd.DataFrame(data)

        # Parsear FechaObservacion
        if 'FechaObservacion' in df.columns:
            df['FechaObservacion'] = pd.to_datetime(
                df['FechaObservacion'],
                format='%Y %b %d %I:%M:%S %p'
            )

        print(f"Se obtuvieron {len(df)} registros")
        return df

    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()


# EJEMPLO: Enero a Marzo 2026, Pereira
from datetime import date

df = consultar_temperatura_aire(
    municipio='PEREIRA',
    fecha_inicio='2026-01-01',
    fecha_fin='2026-03-31'
)

print(df.head())
print(f"\nTotal: {len(df)} registros")
if len(df) > 0:
    print(f"Fechas: {df['FechaObservacion'].min()} a {df['FechaObservacion'].max()}")


# PRUEBA 1: Sin filtros
# print("=" * 60)
# print("PRUEBA 1: Consulta sin filtros")
# print("=" * 60)
df1 = consultar_temperatura_aire()

if len(df1) > 0:
    print("\nResultado:")
    print(df1.head())


df2 = consultar_temperatura_aire(
    municipio='TERUEL',
    fecha_inicio='2023-06-05',
    fecha_fin='2023-06-05'
)
print(len(df2))
# print(df2)

**Función consultar_temperatura_aire**

Consulta datos de temperatura ambiente del aire desde la API pública de datos.gov.co. Permite filtrar por municipio y rango de fechas, devolviendo un DataFrame con los registros encontrados.
Parámetros:

municipio (opcional): Nombre del municipio para filtrar resultados
fecha_inicio y fecha_fin (opcionales): Rango de fechas en formato YYYY-MM-DD
limit (defecto 50000): Número máximo de registros a retornar

Retorna:
DataFrame con las columnas de la API (estación, sensor, fecha, temperatura, ubicación geográfica, etc.), o vacío si no hay resultados.
Cambios principales:

Corrige el formato de fecha en la consulta SoQL: usa ISO 8601 (YYYY-MM-DDTHH:MM:SS) en lugar de texto formateado, compatible con la API
Añade validación y escapado de caracteres especiales en nombres de municipios
Mejora mensajes de error para mostrar respuestas de la API cuando falla
Estandariza nombres de columnas en minúsculas

In [ ]:
import requests
import pandas as pd
from datetime import datetime

def consultar_temperatura_aire(municipio=None, fecha_inicio=None, fecha_fin=None, limit=50000):
    """
    Consulta datos de Temperatura Ambiente del Aire

    Args:
        municipio: str (ej: 'BOGOTÁ')
        fecha_inicio: str 'YYYY-MM-DD' (ej: '2026-01-01')
        fecha_fin: str 'YYYY-MM-DD' (ej: '2026-06-04')
    """
    dataset_id = "sbwg-7ju4"
    url = f"https://www.datos.gov.co/resource/{dataset_id}.json"

    conditions = []

    if municipio is not None:
        # Escapar comillas simples en el municipio
        municipio_escaped = municipio.replace("'", "''")
        conditions.append(f"`Municipio` = '{municipio_escaped}'")

    # Usar formato ISO 8601 para fechas en SoQL
    if fecha_inicio is not None and fecha_fin is not None:
        try:
            inicio = datetime.strptime(fecha_inicio, '%Y-%m-%d')
            fin = datetime.strptime(fecha_fin, '%Y-%m-%d')

            # Formato ISO 8601 para SoQL
            inicio_str = inicio.strftime('%Y-%m-%dT00:00:00')
            fin_str = fin.strftime('%Y-%m-%dT23:59:59')

            conditions.append(f"`FechaObservacion` >= '{inicio_str}'")
            conditions.append(f"`FechaObservacion` <= '{fin_str}'")

            print(f"Rango: {inicio_str} a {fin_str}")

        except ValueError as e:
            print(f"Error en formato fecha: {e}")
            return pd.DataFrame()

    where_clause = " AND ".join(conditions) if conditions else ""
    where_str = f"WHERE {where_clause}" if where_clause else ""

    query = f"""SELECT *
{where_str}
LIMIT {limit}"""

    print(f"Consultando: municipio={municipio}")
    print(f"Query: {query}")  # Debug

    params = {'$query': query}

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()

        data = response.json()

        if len(data) == 0:
            print(f"⚠️ 0 registros encontrados")
            return pd.DataFrame()

        df = pd.DataFrame(data)

        # Parsear FechaObservacion (ISO format)
        if 'fechaobservacion' in df.columns:
            df['fechaobservacion'] = pd.to_datetime(df['fechaobservacion'])

        print(f"✓ Se obtuvieron {len(df)} registros")
        return df

    except requests.exceptions.HTTPError as e:
        print(f"❌ Error HTTP: {e}")
        print(f"Status: {response.status_code}")
        print(f"Respuesta: {response.text[:200]}")
        return pd.DataFrame()
    except Exception as e:
        print(f"❌ Error: {e}")
        return pd.DataFrame()


# Prueba: Sin filtros (para ver qué municipios existen)
# print("=" * 60)
# print("PRUEBA 1: Primeros 5 municipios sin filtros")
# print("=" * 60)
# df1 = consultar_temperatura_aire(limit=100)
# if len(df1) > 0:
#     municipios_unicos = df1['municipio'].unique()[:10]
#     print(f"Municipios encontrados: {municipios_unicos}")
#     print(df1[['municipio', 'fechaobservacion', 'valorobservado']].head())

# Prueba con parámetros (usa un municipio real)
print("\n" + "=" * 60)
print("PRUEBA 2: Con parámetros (PEREIRA)")
print("=" * 60)
df2 = consultar_temperatura_aire(
    municipio='PEREIRA',
    fecha_inicio='2026-01-01',
    fecha_fin='2026-03-31'
)
if len(df2) > 0:
    print(df2[['municipio', 'fechaobservacion', 'valorobservado']].head())